# Tanager Mangrove Mapping - 03 GEDI Fusion

| | |
|---|---|
| **Authors**     | Muhammad Wahyu Ramadhan, Athar Abdurrahman B., Diniyarti |
| **Competition** | Planet Tanager Open Data Competition 2026 |
| **Topic**       | Transferable Mangrove Extent and Biomass Mapping Using Adaptive Spectral Thresholds |
| **Date**        | June 2026 |

---

**Scope:** GEDI L4A footprint loading, spatial join with Tanager indices, AGB regression, wall-to-wall biomass and carbon map for Sangatta.

## 0. Environment Setup

In [ ]:
# Install dependencies (commented out for production)
# !pip install geopandas rasterio scikit-learn matplotlib joblib

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

# ============================================================
# Project root
# ============================================================
ROOT        = Path('..').resolve()
DATA_RAW    = ROOT / 'data' / 'raw'
DATA_PROC   = ROOT / 'data' / 'processed'
OUT_MODELS  = ROOT / 'outputs' / 'models'
OUT_RESULTS = ROOT / 'outputs' / 'results'
OUT_FIGURES = ROOT / 'outputs' / 'figures'

sys.path.insert(0, str(ROOT))
from src.preprocessing import load_geotiff_bands, compute_all_indices
from src.classification import load_model
from src.gedi_utils import (
    load_gedi_footprints,
    spatial_join_gedi_tanager,
    train_agb_regressor,
    predict_wall_to_wall_agb,
    agb_to_carbon,
    save_regressor
)

SCENE_ID = '20250302_030003_92_4001'   # Sangatta
print(f'ROOT         : {ROOT}')
print(f'Scene ID     : {SCENE_ID}')

## 1. Load Tanager Indices + Mangrove Extent

In [ ]:
# ============================================================
# Load GeoTIFF bands and indices
# ============================================================
data    = load_geotiff_bands(str(DATA_PROC), SCENE_ID)
indices = compute_all_indices(data)

# Load mangrove extent from 02_classification.ipynb output
extent_path  = DATA_PROC / f'extent_mangrove_{SCENE_ID}.tif'
with rasterio.open(extent_path) as src:
    extent_map   = src.read(1)

mangrove_mask = extent_map == 1
print(f'Mangrove pixels  : {int(np.sum(mangrove_mask)):,}')

## 2. Load GEDI L4A Footprints

In [ ]:
# ============================================================
# GeoJSON exported from GEE by Athar
# Filters: l4_quality_flag=1, sensitivity>0.95, agbd>0
# ============================================================
gedi_path = DATA_RAW / 'gedi_l4a_sangatta.geojson'
gedi_gdf  = load_gedi_footprints(str(gedi_path))

## 3. Spatial Join — GEDI x Tanager

In [ ]:
# ============================================================
# Extract index values at each GEDI footprint location
# Reference TIF: any processed band for CRS + transform
# ============================================================
ref_tif  = next(DATA_PROC.glob(f'{SCENE_ID}_nir_*.tif'))
joined   = spatial_join_gedi_tanager(gedi_gdf, indices, str(ref_tif))

print(f'\nJoined sample stats:')
print(joined.describe().round(3))

## 4. AGB Regression

In [ ]:
# ============================================================
# RF regressor — weighted by inverse agbd_se
# ============================================================
feature_names = ['NDMI', 'MNDWI', 'MVI', 'SAVI', 'EMI']

agb_reg, agb_metrics = train_agb_regressor(joined, feature_names)
save_regressor(agb_reg, str(OUT_MODELS / f'agb_regressor_{SCENE_ID}.joblib'))

# Save metrics
import json
with open(OUT_RESULTS / f'agb_metrics_{SCENE_ID}.json', 'w') as f:
    json.dump(agb_metrics, f, indent=2)
print(f'\nMetrics saved')

## 5. Wall-to-Wall AGB + Carbon Map

In [ ]:
# ============================================================
# Predict AGB within mangrove extent only
# ============================================================
agb_map    = predict_wall_to_wall_agb(agb_reg, indices, mangrove_mask)
carbon_map = agb_to_carbon(agb_map)

In [ ]:
# ============================================================
# Save AGB + carbon maps as GeoTIFF
# ============================================================
h, w = agb_map.shape

for arr, label in [(agb_map, 'agb'), (carbon_map, 'carbon')]:
    out_path = DATA_PROC / f'{label}_map_{SCENE_ID}.tif'
    with rasterio.open(
        out_path, 'w',
        driver='GTiff', height=h, width=w,
        count=1, dtype='float32',
        crs=data['crs'], transform=data['transform'],
        compress='lzw', nodata=np.nan
    ) as dst:
        dst.write(arr, 1)
    print(f'{label} map saved : {out_path}')

## 6. Visualization

In [ ]:
# ============================================================
# AGB + carbon map side-by-side
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(agb_map, cmap='YlGn', vmin=0, vmax=300)
axes[0].set_title('Above-Ground Biomass (Mg/ha)')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, label='Mg/ha')

im1 = axes[1].imshow(carbon_map, cmap='YlGn', vmin=0, vmax=150)
axes[1].set_title('Carbon Stock (MgC/ha)')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, label='MgC/ha')

plt.suptitle(f'GEDI Fusion — Sangatta ({SCENE_ID})', y=1.02)
plt.tight_layout()
plt.savefig(OUT_FIGURES / f'agb_carbon_map_{SCENE_ID}.png', dpi=150, bbox_inches='tight')
plt.show()